# Análisis Exploratorio del Dataset de Reseñas de productos y cuidado de la piel de Sephora

Este notebook explora el dataset crudo compuesto por el catálogo de productos de cosmética y las valoraciones de sus clientes. Identificaremos valores faltantes ocultos y propondremos un flujo de limpieza, integración de datos y análisis adaptado al contexto del proyecto..

## Configuración del Entorno y la Carga de Datos
Para garantizar la consistencia de este análisis, hemos configurado el entorno de trabajo para que sea dinámico. El código detecta automáticamente si se está ejecutando en la nube (**Google Colab**) o en un entorno local (**Visual Studio Code**), ajustando las rutas de los archivos.

**Primera validación de datos:** En esta etapa procedemos a cargar el catálogo principal (`product_info.csv`) y a unificar los múltiples archivos de comentarios de usuarios (`reviews_*.csv`) en un solo conjunto de datos. Al momento de cargar los archivos, aplicamos una limpieza inicial utilizando el parámetro `na_values`. Esto asegura que los strings vacíos o textos como "NULL" sean interpretados correctamente por Pandas como valores faltantes (`NaN`), preparándolos para nuestro análisis de nulos.

In [ ]:
# Se importan las libterías necesarias para el análisis de datos y visualización
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Se ocultan advertencias para mantener el notebook limpio
warnings.filterwarnings('ignore') # Aveces, las advertencias pueden ser molestas

# Se configuran la visualizacion para que se vea más profesional
sns.set_theme(style='whitegrid') # Esto hace que tenga una cuadricula blanca de fondo
plt.rcParams['figure.figsize'] = (10, 6) # Esto ajusta el tamaño de los graficos

In [ ]:
# Detectar si estamos en Google Colab y cargar los archivos de datos
try:
    import google.colab
    IN_COLAB = True # Si funciona, estamos en la nube (colab)
except ImportError:
    IN_COLAB = False # Si falla, estamos en local (VSC)

# Se configura la ruta de los archivos dependiendo del entorno
if IN_COLAB:
    print('Google Colab detectado. Sube TODOS los arhivos CSV al mismo tiempo (product_info.csv y las 5 reviews_*.csv).')
    from google.colab import files
    uploaded = files.upload() # Esto abre una ventana para seleccionar múltiples archivos
    if not uploaded:
        # Si se cancela la subida, se lanza un error
        raise FileNotFoundError('Debe subir los archivos en Colab antes de ejecutar el notebook.')
    ruta_base = '' # En Colab los archivos quedan en la raíz temporal
else:
    # Si no estamos en Colab, se asume que los archivos están en la ruta local
    ruta_base = '../data/raw/'

# Se definen los nombres exactos de los archivos
archivo_productos = 'product_info.csv'
archivos_reviews = [
    'reviews_0-250.csv',
    'reviews_250-500.csv', 
    'reviews_500-750.csv', 
    'reviews_750-1250.csv', 
    'reviews_1250-end.csv'
]

print('\nIniciando la carga de los datos...')

# Se carga el catálogo de los productos
# El parametro 'na_values' transforma textos vacios o "NULL" en valores falatantes (NaN)
print(f'-> Productos cargados: {df_productos.shape[0]} filas.')
ruta_prod = f"{ruta_base}{archivo_productos}" if ruta_base else archivo_productos
df_productos = pd.read_csv(ruta_prod, na_values=['NULL', 'null', ''])

# Se cargan y se unen las 5 partes de reseñas
lista_reviews = []
for archivo in archivos_reviews:
    ruta_rev = f"{ruta_base}{archivo}" if ruta_base else archivo
    df_temp = pd.read_csv(ruta_rev, na_values=['NULL', 'null', ''])
    lista_reviews.append(df_temp)

df_reviews = pd.concat(lista_reviews, ignore_index=True)
print(f'-> Reseñas unidas: {df_reviews.shape[0]} filas en total.')

# Se unen las reseñas y productos en un solo DataFrame principal
# Usamos 'how="left"' para conservar todas las reseñas y traer la info del producto correspondiente
df_final = pd.merge(df_reviews, df_productos, on='product_id', how='left')

# Se hace una inspección rápida del dataset para entender su estructura
print('\n¡Unión completada exitosamente!')
print(f'Dimensiones del dataset final (df_final): {df_final.shape}') # Esto muestra cuantas filas y columnas tiene el dataset
df_final.head() # Esto muestra las primeras 5 filas del dataset para tener una idea de su contenido

## Diccionario de variables clave
Dado que hemos integrado el catálogo de productos con las reseñas de los usuarios, nuestro dataset ahora cuenta con información tanto del artículo como del perfil del cliente:

**Datos del Producto:**
- `product_id`: Identificador único del producto.
- `product_name`: Nombre comercial del producto.
- `brand_name`: Marca del producto.
- `price_usd`: Precio del producto en dólares.
- `ingredients`: Lista de ingredientes del producto.
- `primary_category` / `secondary_category`: Categorías a las que pertenece el producto.

**Datos de la Reseña y del Usuario:**
- `author_id`: Identificador único del usuario que escribe la reseña.
- `review_text`: Texto completo de la reseña o comentario.
- `rating`: Valoración de 1 a 5 estrellas.
- `is_recommended`: Indica si el usuario recomienda el producto (1.0 = Sí, 0.0 = No).
- `skin_type`: Tipo de piel del usuario (ej. seca, grasa, mixta).
- `skin_tone`: Tono de piel del usuario.
- `eye_color` / `hair_color`: Color de ojos y cabello del usuario.

In [ ]:
# Información general del dataset
print('Información del dataset:')
df_final.info() # Nos muestra el tipo de datos de cada columna y cuántos valores no nulos que tiene

print("\n" + "="*50 + "\n") # Esto es solo para separar visualmente las secciones del notebook

print('\nResumen estadístico de variables numéricas:')
df_final.describe(include='all').T # Esto nos genera estadísticas descriptivas para todas las variables

### Inspección de datos:
Al analizar el resultado de `info()` y `describe()`, descubrimos problemas importantes de calidad en los datos que justifican un preprocesamiento fuerte en nuestro Pipeline:

1. **Columnas completamente vacías:** Las variables `rating`, `view_count` y `dupes` tienen 0 valores no nulos (100% nulos). En su estado actual, no aportan información.
2. **Tipos de datos incorrectos (Data Types):** La variable `price_site` representa el precio, pero Pandas la clasificó como texto (object). Al revisar las estadísticas descriptivas, notamos que los valores incluyen el símbolo `$` (ej. $18.00). Deberemos limpiar este caracter y convertir la columna a formato numérico (float).
3. **Alta proporción de nulos en imágenes:** La columna `img` apenas tiene 1,195 registros válidos de 10,667 (aprox. 89% de datos faltantes), mientras que `shade_img` también presenta valores nulos y un formato complejo.

In [ ]:
# Análisis de valores faltantes y valores ocultos
missing_counts = df.isna().sum().sort_values(ascending=False) # Se calcula la cantidad de nulos por columna
missing_ratio = (df.isna().mean() * 100).round(2) # Se calcula el porcentaje que representan los nulos
missing_summary = pd.DataFrame({'missing_count': missing_counts, 'missing_ratio': missing_ratio}) # Se crea un DataFrame

# Se muestra un resumen de los valores faltantes por columna
print('--- Valores faltantes por columna ---')
print(missing_summary[missing_summary['missing_count'] > 0]) # Se muestra solo las columnas que tienen valores faltantes

# Se visualiza los nulos en un mapa de calor para identificar patrones
plt.figure(figsize=(12, 4))
sns.heatmap(df.isna(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Mapa de calor de valores faltantes')
plt.xlabel('Columnas')
plt.show()

### Conclusiones del Análisis
La tabla y el mapa de calor confirman las observaciones iniciales. El patrón de datos faltantes (las franjas amarillas) nos obliga a tomar decisiones estratégicas de limpieza:

1. **Candidatas a eliminación (100% nulos):** Las variables `rating`, `dupes` y `view_count` están completamente vacías. Deberán ser eliminadas por completo del dataset, ya que no aportan ninguna información.
2. **Variables de imágenes con alta carencia:** `img` (88.7% nulos) y `shade_img` (39.5% nulos). Debido al alto porcentaje de faltantes, evaluaremos eliminarlas o extraer si existe o no la imagen como una variable binaria.
3. **Variables clave incompletas:** `price_site` (32.8% nulos) y `product_type` (10.9% nulos) requerirán estrategias de imputación (rellenar con la media, mediana, moda o valor constante) en el pipeline, ya que son características importantes del producto.

## Análisis de variables categóricas principales

In [ ]:
# Explorar categorías principales
print('Top 12 tipos de producto:')
print(df['product_type'].value_counts().head(12)) # Muestra las frecuencias de la variable

# Se visualiza los tipos de producto más comunes con un gráfico de barras horizontal
plt.figure(figsize=(12, 5))
# Se ocupa countplot para contar la cantidad de productos por tipo
ax = sns.countplot(data=df, y='product_type', order=df['product_type'].value_counts().head(12).index, palette='viridis')
ax.set_title('Top 12 tipos de producto')
ax.set_xlabel('Cantidad')
ax.set_ylabel('Tipo de producto')
plt.tight_layout()
plt.show()

# Hace una separación
print('\n' + '='*50 + '\n')

# Se explora la frecuencia de la variable Marca
print('Top 10 marcas:')
print(df['brand'].value_counts().head(10))

# Sevisualiza las marcas
plt.figure(figsize=(12, 5))
ax = sns.countplot(data=df, y='brand', order=df['brand'].value_counts().head(10).index, palette='magma')
ax.set_title('Top 10 marcas con más productos')
ax.set_xlabel('Cantidad de productos')
ax.set_ylabel('Marca')
plt.tight_layout()
plt.show()

### Análisis de las Variables Categóricas
Al analizar las frecuencias de los tipos de productos y las marcas, se detecto un problema crítico:

1. **Inconsistencia oculta:** Al inspeccionar la salida en texto de `product_type`, descubrimos que todas las categorías traen un salto de línea "insertada" al principio. Si se pasa por alto, los algoritmos podrían fallar al categorizar.
2. **Distribución del catálogo:** El mercado está claramente dominado por productos en formato polvo y productos generales para el rostro. 
3. **Concentración de marcas:** `e.l.f. Cosmetics` y `MAC` son las marcas con mayor volumen de productos en nuestra base de datos.

## Análisis de distribución

In [ ]:
# Se filtran las columnas numéricas para analizar su distribución
numeric_cols = df.select_dtypes(include=['number']).columns.tolist() 
numeric_cols = [col for col in numeric_cols if col != 'ID'] # Se excluye la columna ID 

# Se muestra las variables
print('Variables numéricas detectadas:', numeric_cols)

# Si hay variables numéricas, se gráfica
if numeric_cols:
    n_numeric = len(numeric_cols)
    # Se crea una figura con muchos subgráficos
    fig, axs = plt.subplots(n_numeric, 1, figsize=(12, 5 * n_numeric), squeeze=False)
    axs = axs.flatten()
    # Se gráfica un histograma para cada variable numérica
    for ax, col in zip(axs, numeric_cols):
        # Se ocupa kde=True para mostrar la curva y las barras
        sns.histplot(data=df, x=col, kde=True, ax=ax, color='steelblue')
        ax.set_title(f'Distribución de {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()
# Si no hay variables numéricas, se muestra un mensaje
else:
    print('No se encontraron variables numéricas válidas para el análisis.')

### Distribuciones Numéricas (Comprobación de Errores)
Los histogramas generados nos entregan una confirmación visual que se habia detectado en la fase de inspección:

1. **Gráficos vacíos por ausencia de datos:** Las variables `rating`, `view_count` y `dupes` fueron reconocidas como numéricas, pero sus gráficos están completamente en blanco. Esto ocurre porque tienen un 100% de valores nulos. No hay distribución que graficar.
2. **La variable `price_site` no fue detectada:** Nuestro principal dato numérico (el precio) no aparece en este análisis. Como se vio anteriormente, los símbolos de moneda (`$`) forzaron a Pandas a clasificar esta columna como texto (`object`).

**Justificación del entorno de limpieza:** Estos gráficos vacíos demuestran que es imposible realizar un análisis estadístico profundo (como correlaciones) sin antes someter el dataset a un proceso estricto de limpieza de texto e imputación de nulos.

## Tratamientos de Outliers

In [ ]:
# Se selecciona las columnas detectadas como númericas
cols_outliers = [col for col in numeric_cols if col not in ['ID']]
df_capped = df.copy() # Se crea una copia del DataFrame
for col in cols_outliers: # Se calcula el IQR para cada columna numérica
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1 # Diferencia entre el tercer cuartil y el primer cuartil

    # Se definen los limites
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Se aplica el capping para limitar los valores fuera de los límites sin eliminar registros
    df_capped[col] = df[col].clip(lower, upper)

print('Outliers tratados mediante capping para variables numéricas sin perder registros.')

# Se generan las visualizaciones para comprar
if cols_outliers:
    n_outliers = len(cols_outliers)
    fig, axs = plt.subplots(n_outliers, 2, figsize=(14, 5 * n_outliers), squeeze=False)
    for idx, col in enumerate(cols_outliers):
        # Gráfico de boxplot original
        sns.boxplot(data=df, x=col, ax=axs[idx, 0], color='indianred')
        axs[idx, 0].set_title(f'Boxplot original de {col}')

        # Gráfico de boxplot con capping
        sns.boxplot(data=df_capped, x=col, ax=axs[idx, 1], color='mediumseagreen')
        axs[idx, 1].set_title(f'Boxplot con capping de {col}')

    plt.tight_layout()
    plt.show()
# Si no hay columnas numéricas, se muestra un mensaje
else:
    print('No hay columnas numéricas disponibles para análisis de outliers.')

### Evaluación del Tratamiento de Outliers (IQR)
Hemos programado la lógica para tratar los valores atípicos mediante la técnica de **Capping** (recorte basado en el Rango Intercuartílico). Sin embargo, la ejecución directa sobre los datos crudos vuelve a fracasar por las razones ya mencionadas anteriormente:

1. **Cálculos matemáticos nulos:** Funciones como `.quantile()` retornan `NaN` al evaluar columnas con 100% de valores faltantes (`rating`, `dupes`, `view_count`), lo que resulta en gráficos vacíos.
2. **Exclusión de la variable objetivo:** `price_site` no fue sometida a este proceso porque aún posee un formato de texto debido a sus caracteres especiales.


## Análisis de Correlación

In [ ]:
# Se verifican si existen variables numéricas para calcular la matriz de correlación
if numeric_cols:
    # Se calcula la matriz de correlación de Pearson
    corr_matrix = df[numeric_cols].corr()
    # Si la matriz de correlación está vacía, se avisa
    if corr_matrix.empty:
        print('No hay suficientes variables numéricas para calcular la matriz de correlación.')
    else:
        # Se crea una máscara para ocultar la mitad superior de la matriz (ya que es simétrica)
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        fig, ax = plt.subplots(figsize=(10, 8))
        # Se genera un mapa de calor
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1, ax=ax)
        ax.set_title('Matriz de correlación de variables numéricas')
        # Se ajustan las etiquetas para que se vean mejor
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print('No se encontraron variables numéricas para la matriz de correlación.')

### Análisis de Correlación
El intento de generar una matriz de correlación cierra nuestro diagnóstico del estado actual del dataset:

1. **Ausencia de Relaciones:** Debido a que las variables numéricas actuales están vacías o mal tipadas, no es posible identificar correlaciones (como la relación entre el precio y el rating).
2. **Estado de los Datos:** Contamos con un dataset con un gran potencial (más de 10,000 registros), pero con problemas graves de:
    * **Integridad:** Columnas 100% nulas.
    * **Formato:** Precios con símbolos de moneda y categorías con saltos de línea.
    * **Tipado:** Variables numéricas reconocidas como texto.

## Conclusiones preliminares
Tras realizar el Análisis Exploratorio de Datos (EDA) sobre el dataset de productos de belleza, se han obtenido los siguientes hallazgos críticos que guiarán la fase de preprocesamiento:

1. Estado de la Integridad de los Datos
Variables Críticas Inutilizables: Se identificó que las columnas rating, view_count y dupes presentan un 100% de valores nulos. Estas variables serán descartadas en la fase de limpieza, ya que no aportan información para el modelado.

2. Desafíos de Formato y Tipado (Dirty Data)
Confusión de Tipos: La variable objetivo o principal, price_site, está mal clasificada como texto (object) debido a la presencia de símbolos de moneda ($). Esto impide cualquier análisis estadístico actual.

### Plan de Acción para el Pipeline de Preprocesamiento
Basado en este diagnóstico, el Pipeline a construir deberá ejecutar obligatoriamente los siguientes pasos:

- DropColumns Transformer: Eliminación de variables con 100% de nulos.
- PriceCleaner Transformer: Limpieza de símbolos $ y conversión a float.
- TextStripper: Limpieza de saltos de línea y espacios en blanco en variables categóricas.
- Imputer: Estrategia de llenado para nulos restantes.
- Outlier Handler: Aplicación de Capping mediante IQR (una vez que los datos sean numéricos).
- Encoder & Scaler: Codificación de marcas/tipos y escalamiento de precios.